## Looking for ways to deal with deponent verbs

In [9]:
#quarying active-voice verbs whose lemma ends in μαι.

import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

def normalize_greek(text):
    if not isinstance(text, str):
        return ""
    decomposed = unicodedata.normalize("NFD", text.strip().lower())
    return "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn")

# Resolve treebank directory from this notebook location.
candidate_treebank_dirs = [
    Path("../treebanks/perseus"),
]

treebank_dir = next((p for p in candidate_treebank_dirs if p.exists()), None)

In [10]:
# All unique ACTIVE verbs (any tense/mood) with lemma ending in μαι.
rows_active_mai = []

for xml_file in sorted(treebank_dir.glob("*.xml")):
    root = ET.parse(xml_file).getroot()
    for sentence in root.findall(".//sentence"):
        sentence_id = sentence.get("id")
        for word in sentence.findall("word"):
            postag = (word.get("postag") or "").strip()
            lemma = (word.get("lemma") or "").strip()

            is_active_verb = len(postag) >= 6 and postag.startswith("v") and postag[5] == "a"
            is_mai_lemma = normalize_greek(lemma).endswith("μαι")

            if is_active_verb and is_mai_lemma:
                rows_active_mai.append(
                    {
                        "file": xml_file.name,
                        "sentence_id": sentence_id,
                        "word_id": word.get("id"),
                        "form": word.get("form"),
                        "lemma": lemma,
                        "postag": postag,
                    }
                )

active_mai_tokens = pd.DataFrame(rows_active_mai)

active_mai_unique_lemmas = (
    active_mai_tokens.groupby("lemma", dropna=False)
    .size()
    .reset_index(name="token_count")
    .sort_values("token_count", ascending=False)
    .reset_index(drop=True)
)

print(f"Total matching tokens: {len(active_mai_tokens)}")
print(f"Unique lemmas: {len(active_mai_unique_lemmas)}")

# Full unique-lemma list with counts.
active_mai_unique_lemmas

Total matching tokens: 2001
Unique lemmas: 119


,lemma,token_count
0,ἔρχομαι,891
1,γίγνομαι,180
2,ἐπέρχομαι,84
3,οἴομαι,77
4,εἰσέρχομαι,76
...,...,...
114,συμφράζομαι,1
115,συμπροθυμέομαι,1
116,βούλομαι,1
117,προστήκομαι,1


In [11]:
# Tense-Voice-Mood breakdown for active verbs with lemma ending in μαι.
tense_map = {
    "p": "present",
    "i": "imperfect",
    "f": "future",
    "a": "aorist",
    "r": "perfect",
    "l": "pluperfect",
    "t": "future perfect",
    "-": "not_marked",
}

mood_map = {
    "i": "indicative",
    "s": "subjunctive",
    "o": "optative",
    "m": "imperative",
    "n": "infinitive",
    "p": "participle",
    "-": "not_marked",
}

voice_map = {
    "a": "active",
    "m": "middle",
    "p": "passive",
    "e": "middle/passive",
    "-": "not_marked",
}

pos_tvm = active_mai_tokens.copy()
pos_tvm["tense_code"] = pos_tvm["postag"].astype(str).str[3]
pos_tvm["mood_code"] = pos_tvm["postag"].astype(str).str[4]
pos_tvm["voice_code"] = pos_tvm["postag"].astype(str).str[5]

pos_tvm["tense"] = pos_tvm["tense_code"].map(tense_map).fillna("unknown")
pos_tvm["mood"] = pos_tvm["mood_code"].map(mood_map).fillna("unknown")
pos_tvm["voice"] = pos_tvm["voice_code"].map(voice_map).fillna("unknown")

# Frequency tables
by_tense = pos_tvm["tense"].value_counts().rename_axis("tense").reset_index(name="count")
by_mood = pos_tvm["mood"].value_counts().rename_axis("mood").reset_index(name="count")
by_voice = pos_tvm["voice"].value_counts().rename_axis("voice").reset_index(name="count")
by_tense_mood_voice = (
    pos_tvm.groupby(["tense", "mood", "voice"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Tense counts")
display(by_tense)
print("Mood counts")
display(by_mood)
print("Voice counts")
display(by_voice)
print("Tense-Mood-Voice combinations")
display(by_tense_mood_voice)

Tense counts


,tense,count
0,aorist,1540
1,perfect,321
2,present,105
3,pluperfect,25
4,future,6
5,imperfect,4


Mood counts


,mood,count
0,indicative,913
1,participle,662
2,infinitive,222
3,subjunctive,108
4,optative,63
5,imperative,33


Voice counts


,voice,count
0,active,2001


Tense-Mood-Voice combinations


,tense,mood,voice,count
0,aorist,indicative,active,680
1,aorist,participle,active,491
2,aorist,infinitive,active,175
3,perfect,participle,active,159
4,perfect,indicative,active,116
5,aorist,subjunctive,active,105
6,present,indicative,active,84
7,aorist,optative,active,59
8,perfect,infinitive,active,43
9,aorist,imperative,active,30


In [18]:
# Perfect active tokens (from the already filtered set: active voice + lemma ending in μαι)
perfect_active_mai_tokens = pos_tvm[
    (pos_tvm["tense"] == "perfect") & (pos_tvm["voice"] == "active")
].copy()

# Unique lemmas with counts
perfect_active_mai_lemmas = (
    perfect_active_mai_tokens.groupby("lemma", dropna=False)
    .size()
    .reset_index(name="token_count")
    .sort_values("token_count", ascending=False)
    .reset_index(drop=True)
)

display(perfect_active_mai_tokens.sample(30)) 

,file,sentence_id,word_id,form,lemma,postag,tense_code,mood_code,voice_code,tense,mood,voice
307,tlg0008.tlg001.perseus-grc1.13.tb.xml,1275,20,γεγονέναι,γίγνομαι,v--rna---,r,n,a,perfect,infinitive,active
373,tlg0011.tlg003.perseus-grc1.tb.xml,2383361,17,γεγώς,γίγνομαι,v-srpamn-,r,p,a,perfect,participle,active
635,tlg0012.tlg001.perseus-grc1.tb.xml,2279762,14,ἐγγεγάασιν,ἐγγίγνομαι,v3pria---,r,i,a,perfect,indicative,active
420,tlg0011.tlg004.perseus-grc1.tb.xml,2387393,15,γεγώς,γίγνομαι,v-srpamn-,r,p,a,perfect,participle,active
1583,tlg0016.tlg001.perseus-grc1.1.tb.xml,1215,14,γεγόνασι,γίγνομαι,v3pria---,r,i,a,perfect,indicative,active
130,tlg0007.tlg004.perseus-grc1.tb.xml,330,49,γεγονέναι,γίγνομαι,v--rna---,r,n,a,perfect,infinitive,active
440,tlg0011.tlg005.perseus-grc2.tb.xml,2899546,19,γεγώς,γίγνομαι,v-srpamn-,r,p,a,perfect,participle,active
166,tlg0007.tlg015.perseus-grc1.tb.xml,250,21,γεγονότες,γίγνομαι,v-prpamn-,r,p,a,perfect,participle,active
1993,tlg0543.tlg001.perseus-grc1.tb.xml,975,11,γεγονὸς,γίγνομαι,v-srpana-,r,p,a,perfect,participle,active
1039,tlg0012.tlg002.perseus-grc1.tb.xml,2189081,3,εἰλήλουθα,ἔρχομαι,v1sria---,r,i,a,perfect,indicative,active
